# Wine Quality Classification

**Objective:** Explore physicochemical properties and compare Random Forest, SGD and SVC classifiers.

> Dataset note: included CSVs are synthetic practice data created for this portfolio. Replace with approved public datasets for official internship submission.


## Setup
Run the following cell. Install missing packages with `pip install -r ../../requirements.txt` from the repository root.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
df=pd.read_csv('wine_quality_synthetic.csv'); display(df.quality.value_counts().sort_index())
df.hist(figsize=(14,10)); plt.tight_layout(); plt.show()
sns.heatmap(df.corr(numeric_only=True),cmap='coolwarm'); plt.show()
# Three classes: low <=5, medium=6, high >=7. This reduces sparse exact-score classes.
df['quality_class']=pd.cut(df.quality,bins=[0,5,6,10],labels=['low','medium','high'])
X=df.drop(columns=['quality','quality_class']); y=df.quality_class
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
models={'Random Forest':RandomForestClassifier(n_estimators=200,random_state=42,class_weight='balanced'),
        'SGD':make_pipeline(StandardScaler(),SGDClassifier(random_state=42,class_weight='balanced')),
        'SVC':make_pipeline(StandardScaler(),SVC(class_weight='balanced'))}
rows=[]
for name,m in models.items():
    m.fit(Xtr,ytr); pred=m.predict(Xte); rows.append([name,accuracy_score(yte,pred)])
    print(name,classification_report(yte,pred))
    sns.heatmap(confusion_matrix(yte,pred,labels=['low','medium','high']),annot=True,fmt='d'); plt.title(name); plt.show()
display(pd.DataFrame(rows,columns=['Model','Accuracy']).sort_values('Accuracy',ascending=False))
rf=models['Random Forest']; pd.Series(rf.feature_importances_,index=X.columns).sort_values().plot(kind='barh',title='Random Forest feature importance'); plt.show()
print('Class imbalance can reduce recall for rare quality groups; stratification and class weights help. Choose by class-wise F1/recall, not accuracy alone.')
